In [1]:
import awswrangler as awr
import pandas as pd
import os
import openpyxl

### GERANDO dataframes

In [9]:
#gerando todos os boletos pagos
caminho_query = r"C:\Users\raphael.almeida\Documents\Processos\placas_adimplentes\sql\faturas_baixadas.sql"
with open(caminho_query,'r') as arquivo_query:
    query = arquivo_query.read()
df_faturas_baixadas = awr.athena.read_sql_query(query, database='silver')

In [11]:
#gerando todos os boletos prevencimento
caminho_query = r"C:\Users\raphael.almeida\Documents\Processos\placas_adimplentes\sql\faturas_posvencimento.sql"
with open(caminho_query,'r') as arquivo_query:
    query = arquivo_query.read()
df_faturas_prevencimento = awr.athena.read_sql_query(query, database='silver')

In [12]:
#gerando todos os boletos posvencimento
caminho_query = r"C:\Users\raphael.almeida\Documents\Processos\placas_adimplentes\sql\faturas_prevencimento.sql"
with open(caminho_query,'r') as arquivo_query:
    query = arquivo_query.read()
df_faturas_posvencimento = awr.athena.read_sql_query(query, database='silver')

### MODULANDO dataframes (com as categorias necessárias e concatenando)

In [ ]:
# Gerar uma lista com os ponteiros dos boletos pagos para usar no isin
# Gerar um set de tuplas com (ponteiro, empresa, conjunto) dos boletos pagos
boletos_pagos = set(
    zip(
        df_faturas_baixadas['ponteiro'],
        df_faturas_baixadas['empresa'],
        df_faturas_baixadas['conjunto']
    )
)
boletos_pagos

{(np.int64(419421), 'Viavante', np.int64(17406)),
 (np.int64(271293), 'Stcoop', np.int64(17031)),
 (np.int64(473692), 'Viavante', np.int64(21162)),
 (np.int64(260252), 'Stcoop', np.int64(16308)),
 (np.int64(2647151), 'Tag', np.int64(9483)),
 (np.int64(230824), 'Stcoop', np.int64(14663)),
 (np.int64(476470), 'Viavante', np.int64(21523)),
 (np.int64(2680761), 'Tag', np.int64(5840)),
 (np.int64(2604030), 'Tag', np.int64(6398)),
 (np.int64(270256), 'Stcoop', np.int64(16908)),
 (np.int64(442526), 'Viavante', np.int64(19546)),
 (np.int64(387913), 'Viavante', np.int64(15842)),
 (np.int64(237208), 'Stcoop', np.int64(14935)),
 (np.int64(2540899), 'Tag', np.int64(944)),
 (np.int64(243859), 'Stcoop', np.int64(15271)),
 (np.int64(470520), 'Viavante', np.int64(21399)),
 (np.int64(258866), 'Stcoop', np.int64(16359)),
 (np.int64(2547219), 'Tag', np.int64(1629)),
 (np.int64(2532009), 'Segtruck', np.int64(140379)),
 (np.int64(442766), 'Viavante', np.int64(19178)),
 (np.int64(2639030), 'Tag', np.int64(8

In [ ]:
#gerando df_faturas_inadimplentes
df_faturas_inadimplentes = df_faturas_posvencimento[
    ~df_faturas_posvencimento.apply(
        lambda row: (row['ponteiro'], row['empresa'], row['conjunto']) in boletos_pagos, axis=1
    )
]

In [31]:
# a pedido do setor de COBRANÇAS, retirar associado: APROSSIL - ASSOCIACAO DE PROPRIETARIOS DE CAMINHOES DO SUL D
# a pedido do setor de COBRANÇAS, retirar associado: TESTE
df_inadimp_atual = df_faturas_inadimplentes[
    ~(
        (df_faturas_inadimplentes['empresa'].isin(['Viavante', 'Stcoop', 'Segtruck'])) &
        (df_faturas_inadimplentes['associado'] == "APROSSIL - ASSOCIACAO DE PROPRIETARIOS DE CAMINHOES DO SUL D")
    )
]
df_inadimp_atual = df_inadimp_atual[~df_inadimp_atual['associado'].str.contains('TESTE', na=False)]

In [32]:
df_faturas_aberto = df_faturas_posvencimento[~df_faturas_posvencimento['ponteiro'].isin(boletos_pagos)]
df_faturas_aberto = df_faturas_aberto.drop_duplicates(subset=['ponteiro', 'conjunto', 'matricula', 'empresa'], keep='first')

In [33]:
df_inadimp_atual["situacao"] = "inadimplente"
df_faturas_aberto["situacao"] = "aberto"
df_faturas_baixadas["situacao"] = "baixado"
df_faturas = pd.concat([df_inadimp_atual, df_faturas_aberto, df_faturas_baixadas])

In [36]:
caminho_pasta = r'C:\Users\raphael.almeida\Documents\Projetos\Relatório de Faturas'
caminho_arquivo = os.path.join(caminho_pasta, 'relatorio_faturas.xlsx')
os.makedirs(caminho_pasta, exist_ok=True)
if os.path.exists(caminho_arquivo):
    os.remove(caminho_arquivo)
    print("Arquivo antigo removido, iniciando carregamento...")
df_faturas.to_excel(caminho_arquivo, engine='openpyxl', index=False, sheet_name='faturas')
print(f"Arquivo Excel salvo com sucesso em: {caminho_arquivo}")

Arquivo antigo removido, iniciando carregamento...
Arquivo Excel salvo com sucesso em: C:\Users\raphael.almeida\Documents\Projetos\Relatório de Faturas\relatorio_faturas.xlsx
